# SI — dSTORM of microtubules in a HeLa cell (Ximea)

**Makes up:** SI `HeLa_STORM` (super-resolved render with box, zoom with two line profiles, the
profiles fitted with two Gaussians) and `HeLa_FRC` (FRC curve and FIRE resolution; published:
86.5 nm).

**Data:** BioStudies `S-BIAD3210`,
`2026_Multicolour_Paper/Ximea/HeLaCellSTORM/20250523_HeLa_STORM/Cell4_HILO_190mW_638_ximea638_setting/Lp638_190_mw_40ms_exosure_HILO_1/`
— raw Ximea TIFFs (25 files, ~99 GB) plus the acquisition's MicroManager metadata. The
deposition holds raw data only, so this notebook fits it first.

The pipeline (fit → QC → AIM drift correction → linking → fiducial removal → 8× render → FRC) is
the one used for Figure 5's DNA-PAINT data, single channel; settings from the analysis notebooks
(`hela_tubulin/HeLa_Tubulin_Analysis`, `HeLa_Tubulin_PostAnalysis`).

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only the first TIFF (~4 GB, plus the acquisition's 1.8 GB metadata file), enough to run the
whole pipeline end to end, with far fewer localisations than the published figure (so a worse
FIRE value). `FAST_MODE = False` downloads and fits the full acquisition (~99 GB; many hours of
fitting; run in `tmux`/`screen`). The box and line profiles are in camera pixels of the full
acquisition; in `FAST_MODE` they may be sparse.

---
## 0 — Imports

In [ ]:
import json
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from skimage.draw import line as bresenham_line

# _biostudies_download.py lives in the parent folder (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import render
from pyS3M import FiducialDetection
from pyS3M.FRCFunctions import fire
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions
from pyS3M.LinkingFunctions import link_localisations

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
plotter = PlottingBase.PublicationPlotter(dark_background=False)

FD = FiducialDetection.FiducialDetector()

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF only (~4 GB, plus the acquisition's 1.8 GB metadata file, illustrative). False: all 25 TIFFs (~99 GB, the published data).

PIXEL_SIZE_NM = 69.0   # Ximea camera pixel

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "FigureSI_HeLa_STORM"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_DIR at a drive with room for the download. Keep one folder per acquisition and
# per FAST_MODE setting: the fit reads every TIFF inside it.
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

gain_map   = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map = IO.read_tiff(CALIB_DIR / "offset.tif")
variance   = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe        = IO.read_tiff(CALIB_DIR / "rqe.tif")

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Downloads the first TIFF (`FAST_MODE`) or the whole series, plus the acquisition's MicroManager
`*_metadata.txt` in both cases: it holds the camera ROI the frames were cropped to, which the fit
needs to crop the calibration maps. Re-running is safe: complete files are skipped and interrupted
ones resume.

In [ ]:
REMOTE_FOLDER = ("2026_Multicolour_Paper/Ximea/HeLaCellSTORM/20250523_HeLa_STORM/"
                 "Cell4_HILO_190mW_638_ximea638_setting/Lp638_190_mw_40ms_exosure_HILO_1")

downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, first_file_only=FAST_MODE)

# Camera ROI of the acquisition (the fitted xc/yc are relative to its corner).
roi_x, roi_y, width, height = IO.metadata_reader_imageJ(downloaded["metadata"][0])
print(f"ROI: x={roi_x} y={roi_y}, {width} x {height} px")

---
## 3 — Spot detection and fitting

The paper's settings for this acquisition: pfa 1e-4, sigma 1.5, fraction_true 0.2, peak
wavelength 0.7 um, 12 px ROI. `fit_imaging_data` treats the TIFFs as one series and writes
`Localisations.h5` into `DATA_DIR`. A record of what was fitted is saved beside it, so re-running
the notebook reuses the fit unless the files or settings change (or `REFIT = True`).

In [ ]:
FIT_PARAMS = dict(pfa=1e-4, sigma=1.5, fraction_true=0.2, peak_wavelength=0.7, ROI_size=12, NA=1.49)

# Quick single-frame check of the spot detection before the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    DATA_DIR, pfa=FIT_PARAMS["pfa"], sigma=FIT_PARAMS["sigma"],
    fraction_true=FIT_PARAMS["fraction_true"], frame_index=10,
    variance=variance, read_noise=read_noise, rqe=rqe, offset_map=offset_map,
    gain_map=gain_map, peak_wavelength=FIT_PARAMS["peak_wavelength"],
)
plt.show()

In [ ]:
REFIT = False

fit_file = DATA_DIR / "Localisations.h5"
fit_record_file = DATA_DIR / "Localisations_fit_record.json"
fit_record = {"images": [p.name for p in downloaded["images"]], **FIT_PARAMS}

previous = json.loads(fit_record_file.read_text()) if fit_record_file.exists() else None
if REFIT or not fit_file.exists() or previous != fit_record:
    SupRes_F.fit_imaging_data(
        DATA_DIR, smoothing_function, gain_map, offset_map, rqe, read_noise,
        variance=variance, image_type=".tif", **FIT_PARAMS,
    )
    fit_record_file.write_text(json.dumps(fit_record, indent=1))
else:
    print(f"Reusing {fit_file} (same files and settings)")

---
## 4 — Quality-control filters

The cuts used for the paper, copied from the analysis notebook (sizes in nm, converted to camera px).

In [ ]:
loc_data = pd.read_hdf(fit_file)
print(f"Raw localisations: {len(loc_data):,}")

loc_data = loc_data[loc_data["xc_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["yc_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["s_y_err"] < 30 / PIXEL_SIZE_NM]
loc_data = loc_data[(loc_data["s_x"] > 100 / PIXEL_SIZE_NM) & (loc_data["s_x"] < 175 / PIXEL_SIZE_NM)]
loc_data = loc_data[(loc_data["s_y"] > 100 / PIXEL_SIZE_NM) & (loc_data["s_y"] < 175 / PIXEL_SIZE_NM)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.5]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[~((loc_data["spot_snr"] < 2) & (loc_data["photons"] < 500)
                      & (loc_data["spot_matched_filter_response"] < 100))]
loc_data = loc_data[loc_data["spot_background_std"] > 10]
loc_data = loc_data[loc_data["chi_sqr"] < 1.5]
print(f"After QC: {len(loc_data):,}")

---
## 5 — Drift correction (AIM) and linking

Localisations of one molecule in consecutive frames (within the median localisation error,
up to 2 dark frames) are merged into one.

In [ ]:
drift_corrector = Drift_Correction_Functions()

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(loc_data["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

corrected_locs, drift_result = drift_corrector.undrift(
    locs=loc_data.to_records(index=False),
    info=info,
    method="aim",
    segmentation=20,
    intersect_d=20 / PIXEL_SIZE_NM,
    roi_r=60 / PIXEL_SIZE_NM,
)
corrected_locs = pd.DataFrame(corrected_locs)

plt.plot(drift_result.drift_x * PIXEL_SIZE_NM, label="x")
plt.plot(drift_result.drift_y * PIXEL_SIZE_NM, label="y")
plt.xlabel("frame"); plt.ylabel("drift / nm"); plt.legend()
plt.show()

# Frame count from the fitted data, so FAST_MODE (first file only) gets the right number.
n_frames = int(loc_data["frame"].max()) + 1
link_r = 0.5 * (np.median(corrected_locs["xc_err"]) + np.median(corrected_locs["yc_err"]))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)
print(f"Linked localisations: {len(linked):,}")

---
## 6 — Remove fiducials

Fiducial markers stay on for most of the movie. Bright, dense regions of a 1x render that hold
localisations in at least 10% of frames are removed.

In [ ]:
info = [{"Width": width, "Height": height, "Frames": int(linked["frame"].max()), "Pixelsize": PIXEL_SIZE_NM}]
image_1x = render.render(locs=linked.to_records(index=False), info=info, blur_method="smooth", oversampling=1)[1]

region_centers, binary_mask, threshold, _ = drift_corrector.detect_high_density_regions_from_image(
    smoothed_image=image_1x,
    histogram_bins=821,
    threshold_percentile=99.9,
    pixelsize=PIXEL_SIZE_NM,
    title="Fiducial detection",
)
selected_puncta, _ = drift_corrector.select_puncta_from_regions(
    locs=linked.to_records(index=False),
    region_centres=region_centers,
    binary_mask=binary_mask,
    pixelsize=PIXEL_SIZE_NM,
    selection_box_size_nm=350.0,
    min_localisations_per_region=int(0.1 * info[0]["Frames"]),
    title="Fiducial selection",
    create_plot=True,
)
linked_fremoved = pd.DataFrame(FD.remove_puncta_locs(linked.to_records(index=False), selected_puncta))
print(f"{len(linked) - len(linked_fremoved):,} fiducial localisations removed, {len(linked_fremoved):,} left")

IO.write_h5_database(df=linked_fremoved, filepath=DATA_DIR / "Localisations_Undrifted_Linked_FiducialsRemoved.h5",
                     normalise_photons=False)

---
## 7 — Super-resolved render with the zoom box

8× oversampled, Gaussian-blurred by each localisation's precision. The published view is cropped
(`CROP_SR`, SR px: rows 300 to −300, columns 800 to −2000 of the full render); the box marks the
zoom (camera px).

In [ ]:
OVERSAMPLING = 8
SR_PX_NM = PIXEL_SIZE_NM / OVERSAMPLING
CROP_SR = (slice(300, -300), slice(800, -2000))   # rows, cols of the full render (SR px)
BOX = dict(x0=500, y0=450, w=150, h=150)            # zoom, camera px

_, image = render.render(linked_fremoved.to_records(index=False), oversampling=OVERSAMPLING,
                         viewport=((0, 0), (height, width)), blur_method="gaussian")
vmin = np.percentile(image, 1)

fig, ax = plotter.one_column_plot(height=2)
plotter.image_plot(ax, image[CROP_SR], cmap="hot", origin="upper", vmin=vmin, vmax=np.percentile(image, 99.6),
                   scalebar=True, pixelsize=SR_PX_NM, scalebarsize=10000, scalebarlabel="10 μm")
ax.add_patch(patches.Rectangle((BOX["x0"] * OVERSAMPLING - CROP_SR[1].start, BOX["y0"] * OVERSAMPLING - CROP_SR[0].start),
                               BOX["w"] * OVERSAMPLING, BOX["h"] * OVERSAMPLING,
                               linewidth=0.75, edgecolor="white", facecolor="none"))
plt.savefig(FIG_DIR / "Exemplar_SuperRes_Box.svg", dpi=600, format="svg")
plt.show()

---
## 8 — Zoom and line profiles

Two lines drawn by hand on the published zoom ((row, col) end points, camera px within the box).
The render's intensity along each is binned (34.5 nm bins) and fitted with two Gaussians, giving
the separation of the two microtubule edges and each peak's FWHM.

In [ ]:
LINES = [((54, 20), (64, 40)), ((50, 80), (70, 90))]   # solid, dashed

zoom = image[BOX["y0"] * OVERSAMPLING:(BOX["y0"] + BOX["h"]) * OVERSAMPLING,
             BOX["x0"] * OVERSAMPLING:(BOX["x0"] + BOX["w"]) * OVERSAMPLING]

fig, ax = plotter.one_column_plot()
plotter.image_plot(ax, zoom, cmap="hot", origin="upper", vmin=vmin, vmax=np.percentile(image, 99.8),
                   scalebar=True, pixelsize=SR_PX_NM, scalebarsize=1000, scalebarlabel="1 μm")
line_pixels = []
for (r0, c0), (r1, c1) in LINES:
    rows, cols = bresenham_line(r0 * OVERSAMPLING, c0 * OVERSAMPLING, r1 * OVERSAMPLING, c1 * OVERSAMPLING)
    line_pixels.append((rows, cols))
    ax.plot(cols, rows, color="white", linewidth=1.5, ls="-" if len(line_pixels) == 1 else "--")
plt.savefig(FIG_DIR / "Exemplar_SuperRes_Zoomin.svg", dpi=600, format="svg")
plt.show()

In [ ]:
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))


def double_gaussian(x, A1, mu1, s1, A2, mu2, s2, offset):
    return gaussian(x, A1, mu1, s1) + gaussian(x, A2, mu2, s2) + offset


FWHM_PER_SIGMA = 2 * np.sqrt(2 * np.log(2))
BIN_NM = PIXEL_SIZE_NM / 2

fig, axs = plotter.one_column_plot(npanels=2, ratios=[1, 1], height=2, width=2)
for ax, (rows, cols) in zip(axs, line_pixels):
    values = zoom[rows, cols]
    dist = np.concatenate([[0.0], np.cumsum(np.hypot(np.diff(rows.astype(float)), np.diff(cols.astype(float))))]) * SR_PX_NM
    bins = np.arange(dist[0], dist[-1] + BIN_NM, BIN_NM)
    sums, edges = np.histogram(dist, bins=bins, weights=values)
    counts, _ = np.histogram(dist, bins=bins)
    means = np.where(counts > 0, sums / np.maximum(counts, 1), 0.0)
    centres = (edges[:-1] + edges[1:]) / 2
    ax.bar(centres, means, width=BIN_NM * 0.75, color="#888", edgecolor="k", linewidth=0.4)

    peaks, props = find_peaks(means, distance=3, prominence=means.max() * 0.2)
    if len(peaks) >= 2:
        p1, p2 = peaks[np.argsort(props["prominences"])[-2:]]
        p0 = [means[p1], centres[p1], 2 * BIN_NM, means[p2], centres[p2], 2 * BIN_NM, 0.0]
        try:
            popt, pcov = curve_fit(double_gaussian, centres, means, p0=p0, bounds=(0, np.inf), maxfev=5000)
            perr = np.sqrt(np.diag(pcov))
            x_fit = np.linspace(centres[0], centres[-1], 500)
            fwhm = FWHM_PER_SIGMA * popt[[2, 5]]
            ax.plot(0, 0, alpha=0, label=f"PS = {abs(popt[4] - popt[1]):.1f} ± {np.hypot(perr[1], perr[4]):.1f} nm")
            ax.plot(x_fit, gaussian(x_fit, *popt[0:3]) + popt[6], "r--", linewidth=0.75, label=f"FWHM = {fwhm[0]:.0f} nm")
            ax.plot(x_fit, gaussian(x_fit, *popt[3:6]) + popt[6], "g--", linewidth=0.75, label=f"FWHM = {fwhm[1]:.0f} nm")
            print(f"peak separation {abs(popt[4] - popt[1]):.1f} +/- {np.hypot(perr[1], perr[4]):.1f} nm, "
                  f"FWHM {fwhm[0]:.0f} and {fwhm[1]:.0f} nm")
        except RuntimeError:
            print("double-Gaussian fit did not converge")
    else:
        print("fewer than two peaks along this line (expected in FAST_MODE)")
    ax.legend(loc="best", fontsize=6)
    ax.set_xlim([400, 1500])
    ax.set_yticklabels([])
    ax.grid(lw=0.5, alpha=0.5, ls="--", color="gray")
axs[0].set_xticklabels([])
axs[1].set_xlabel("distance / nm", fontsize=8)
plt.savefig(FIG_DIR / "Superres_cutthroughs.svg", dpi=600, format="svg")
plt.show()

---
## 9 — FRC and FIRE resolution

Fourier ring correlation between two halves of the localisations, split by blocks of frames
(50 blocks, averaged over 20 random splits), on an 8× grid. FIRE = the resolution where the FRC
falls to 1/7 (published: 86.5 nm, 82.4–90.6 nm).

In [ ]:
def frc_curve(locs):
    """FIRE resolution and FRC curve of one set of localisations (camera px, with frame).

    Passing the frame column makes fire() split the two halves by time blocks, not row order.
    """
    xyf = locs[["xc", "yc", "frame"]].to_numpy(dtype=float)
    xyf[:, 0] -= xyf[:, 0].min()
    xyf[:, 1] -= xyf[:, 1].min()
    nx = int(xyf[:, 0].max()) + 1
    ny = int(xyf[:, 1].max()) + 1
    res, frc, res_hi, res_lo = fire(positions=xyf, nx=nx, ny=ny, zoom=FRC_ZOOM, n_blocks=50, reps=20,
                                    pixel_size_nm=PIXEL_SIZE_NM)
    # Spatial frequency of each FRC bin in cycles/nm, kept below the SR-pixel Nyquist limit.
    q = np.arange(len(frc)) / (max(nx, ny) * FRC_ZOOM)   # cycles per SR pixel
    keep = (q > 0) & (q < 0.6)
    q_nm = q / (PIXEL_SIZE_NM / FRC_ZOOM)
    return dict(res=res, res_hi=res_hi, res_lo=res_lo, q_nm=q_nm[keep], frc=frc[keep])


def plot_frc(ax, curve, label, color, span_color):
    ax = plotter.line_plot(ax, curve["q_nm"], curve["frc"], label=label, color=color)
    ax.axvline(1 / curve["res"], color=span_color, ls="-", lw=1, label=f"FIRE = {curve['res']:.0f} nm")
    ax.axvspan(1 / curve["res_lo"], 1 / curve["res_hi"], alpha=0.15, color=span_color)
    return ax


FRC_ZOOM = 8   # SR pixels per camera pixel

In [ ]:
frc_hela = frc_curve(linked_fremoved)
print(f"FIRE resolution: {frc_hela['res']:.1f} nm ({frc_hela['res_hi']:.1f}-{frc_hela['res_lo']:.1f} nm)")

fig, ax = plotter.two_column_plot()
ax = plot_frc(ax, frc_hela, "FRC (mean)", "black", "crimson")
ax.axhline(1 / 7, color="gray", ls="--", lw=1, label="1/7 threshold")
ax.set_xlabel("1 / spatial frequency (nm)", fontsize=8)
ax.set_ylabel("FRC", fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xscale("log")
tick_nm = [1000, 500, 200, 100, 50, 30]
ax.set_xticks([1 / d for d in tick_nm])
ax.set_xticklabels([f"{d} nm" for d in tick_nm])
ax.set_xlim(1 / 500, 1 / 15)
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "FRC_HeLa.svg", dpi=600, format="svg")
plt.show()